<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


- Install the additional package requirements for this bonus notebook by uncommenting and running the following cell:

In [1]:
# pip install -r requirements-extra.txt

# Comparing Various Byte Pair Encoding (BPE) Implementations

<br>
&nbsp;

## 1. Using BPE from `tiktoken`

In [2]:
from importlib.metadata import version

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [3]:
import tiktoken

tik_tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, world. Is this-- a test?"

In [ ]:
integers = tik_tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)


[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [5]:
strings = tik_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


In [6]:
print(tik_tokenizer.n_vocab)

50257


<br>
&nbsp;

## 2. Using the original BPE implementation used in GPT-2

In [7]:
from bpe_openai_gpt2 import get_encoder, download_vocab

In [8]:
download_vocab()

Fetching encoder.json: 1.04Mit [00:01, 880kit/s]                                                    
Fetching vocab.bpe: 457kit [00:00, 580kit/s]                                                        


In [9]:
orig_tokenizer = get_encoder(model_name="gpt2_model", models_dir=".")

In [10]:
integers = orig_tokenizer.encode(text)

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [29]:
strings = orig_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


<br>
&nbsp;

## 3. Using the BPE via Hugging Face transformers

In [19]:
import transformers

transformers.__version__

'4.57.3'

In [20]:
from transformers import GPT2Tokenizer

hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [22]:
hf_tokenizer(strings)

{'input_ids': [15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [21]:
hf_tokenizer(strings)["input_ids"]

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

In [24]:
from transformers import GPT2TokenizerFast, AutoTokenizer

hf_tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")
hf_auto_tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [26]:
hf_tokenizer_fast(strings)["input_ids"]

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

In [27]:
hf_auto_tokenizer(strings)

{'input_ids': [15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

<br>
&nbsp;

## 4. Using my own from-scratch BPE tokenizer

In [35]:
%pip install nbformat


   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   -------------------- ------------------- 1/2 [nbformat]
   ---------------------------------------- 2/2 [nbformat]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import os
import sys
import io
import nbformat
import types

def import_from_notebook():
    def import_definitions_from_notebook(fullname, names):
        current_dir = os.getcwd()
        path = os.path.join(current_dir, "..", "05_bpe-from-scratch", fullname + ".ipynb")
        path = os.path.normpath(path)

        # Load the notebook
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")

        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # Create a module to store the imported functions and classes
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod

        # Go through the notebook cells and only execute function or class definitions
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                for name in names:
                    # Check for function or class definitions
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        exec(cell_code, mod.__dict__)
        return mod

    fullname = "bpe-from-scratch"
    names = ["BPETokenizerSimple"]

    return import_definitions_from_notebook(fullname, names)

In [37]:
importted_module = import_from_notebook()
importted_module

<module 'bpe-from-scratch'>

In [39]:

imported_module = import_from_notebook()
BPETokenizerSimple = getattr(imported_module, "BPETokenizerSimple", None)

tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=os.path.join("gpt2_model", "encoder.json"),
    bpe_merges_path=os.path.join("gpt2_model", "vocab.bpe")
)

In [40]:
integers = tokenizer_gpt2.encode(text)

print(integers)

[1212, 318, 617, 2420]


<br>
&nbsp;

## 5. A quick performance benchmark

In [43]:
with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [44]:
raw_text[:100]

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g'

&nbsp;
### 5.1 Original OpenAI GPT-2 tokenizer

In [61]:
%timeit orig_tokenizer.encode(raw_text)

6.82 ms ± 123 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


&nbsp;
### 5.2 Tiktoken OpenAI GPT-2 tokenizer

In [58]:
%timeit tik_tokenizer.encode(raw_text)

1.59 ms ± 13.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


&nbsp;
### 5.3 Hugging Face OpenAI GPT-2 tokenizer

In [59]:
%timeit hf_tokenizer(raw_text)["input_ids"]

16.7 ms ± 305 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [60]:
%timeit hf_tokenizer_fast(raw_text)["input_ids"]

6.37 ms ± 179 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [62]:
%timeit hf_auto_tokenizer(raw_text, max_length=5145, truncation=True)["input_ids"]

6.41 ms ± 233 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [50]:
%timeit hf_tokenizer(raw_text, max_length=5145, truncation=True)["input_ids"]

17.2 ms ± 693 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [51]:
%timeit hf_tokenizer_fast(raw_text)["input_ids"]

7.12 ms ± 580 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [54]:
%timeit hf_tokenizer_fast(raw_text, max_length=5145, truncation=True)["input_ids"]

6.55 ms ± 431 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


&nbsp;
### 5.4 My own GPT-2 tokenizer (for educational purposes)

In [63]:
%timeit tokenizer_gpt2.encode(raw_text)

23 ms ± 602 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
